# Session 3: Training and Evaluating an FNN Language Model with Real Text Data

This notebook covers **Session 3 only**.

Goal: train a simple Feedforward Neural Network for next-word prediction using a larger real-world text dataset.

We use the **20 Newsgroups** dataset from scikit-learn. It contains real posts from discussion newsgroups, so the same language-model pipeline can be seen beyond a tiny toy corpus.

Topics:

1. Load a real-world text dataset
2. Prepare next-word prediction data
3. Review `X`, `y`, vocabulary, and target classes
4. Train/test split
5. Introduce `MLPClassifier`
6. Train the FNN model
7. Evaluate exact accuracy and top-5 accuracy
8. Compare with a simple baseline
9. Predict one next word
10. Inspect prediction probabilities
11. Create helper functions
12. Test different contexts
13. Generate text
14. Limitations and bridge to RNNs/Transformers

## 1. Load a Real-World Text Dataset

The previous sessions used a small text example so every step was easy to inspect.

Now we use more realistic text.

The **20 Newsgroups** dataset contains real discussion posts. We will use four categories:

- `sci.space`
- `rec.autos`
- `comp.graphics`
- `rec.sport.baseball`

The first run may take a little time because Colab may need to download the dataset.

In [ ]:
import re
import warnings
import numpy as np
from collections import Counter

from scipy.sparse import lil_matrix
from sklearn.datasets import fetch_20newsgroups
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier


warnings.filterwarnings("ignore", category=ConvergenceWarning)


categories = [
    "sci.space",
    "rec.autos",
    "comp.graphics",
    "rec.sport.baseball",
]

dataset = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),
)

print("Number of documents:", len(dataset.data))
print("Categories:", dataset.target_names)
print()
print("Example document preview:")
print(dataset.data[0][:700])

## 2. Tokenize and Clean the Text

We use the same beginner tokenization idea:

```text
one word = one token
```

We also:

- convert text to lowercase,
- keep alphabetic word tokens only,
- remove very short documents.

In [ ]:
def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z]+", text)


documents_tokens = []

for document in dataset.data:
    tokens = tokenize(document)
    if len(tokens) >= 5:
        documents_tokens.append(tokens)

all_tokens = [token for document_tokens in documents_tokens for token in document_tokens]

print("Usable documents:", len(documents_tokens))
print("Total tokens:", len(all_tokens))
print("First 40 tokens:")
print(all_tokens[:40])

## 3. Build a Practical Vocabulary

Real text has many unique words.

If we keep every word, the feature vector becomes too large for a short notebook exercise.

So we keep the **600 most common words** as the context vocabulary.

This is a common practical idea: limit vocabulary size to keep training manageable.

In [ ]:
max_context_vocab_size = 600

token_counts = Counter(all_tokens)
most_common_context_words = [word for word, count in token_counts.most_common(max_context_vocab_size)]

vocabulary = sorted(most_common_context_words)
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}

print("Context vocabulary size:", len(vocabulary))
print("Top 30 words by frequency:")
print(token_counts.most_common(30))

## 4. Choose Target Classes

For next-word prediction, the model chooses a target word class.

Real-world text has too many possible next words, so we keep the **120 most common target words**.

This keeps the task realistic but still trainable in a short notebook session.

In [ ]:
max_target_classes = 120

target_words = sorted([word for word, count in token_counts.most_common(max_target_classes)])
target_to_id = {word: index for index, word in enumerate(target_words)}
id_to_target = {index: word for word, index in target_to_id.items()}

print("Number of target classes:", len(target_words))
print("Target class examples:")
print(target_words[:40])

## 5. Create Context-Target Pairs

We use:

```python
context_size = 2
```

That means:

```text
two previous words -> next word
```

To keep training stable:

- context words must be in the 600-word context vocabulary,
- target word must be in the 120 target classes,
- we use up to 12,000 examples so the notebook runs quickly.

In [ ]:
context_size = 2
max_examples = 12000

pairs = []

for document_tokens in documents_tokens:
    for i in range(len(document_tokens) - context_size):
        context = document_tokens[i:i + context_size]
        target = document_tokens[i + context_size]

        context_is_known = all(word in word_to_id for word in context)
        target_is_known = target in target_to_id

        if context_is_known and target_is_known:
            pairs.append((context, target))

        if len(pairs) >= max_examples:
            break

    if len(pairs) >= max_examples:
        break

print("Number of usable context-target pairs:", len(pairs))
print()

for context, target in pairs[:20]:
    print(context, "->", target)

## 6. Build `X` and `y`

For real data, `X` can become large.

Instead of a dense NumPy array, we use a **sparse matrix**.

Why?

One-hot vectors contain mostly zeros.

A sparse matrix stores only the nonzero values, which saves memory.

The idea is the same as Session 2:

```text
context words -> one-hot context vector -> X
target word -> target class ID -> y
```

In [ ]:
def context_to_sparse_row(row_index, context, X, word_to_id, context_size):
    vocab_size = len(word_to_id)

    for position, word in enumerate(context):
        word_id = word_to_id[word]
        feature_index = position * vocab_size + word_id
        X[row_index, feature_index] = 1


num_rows = len(pairs)
num_columns = context_size * len(vocabulary)

X = lil_matrix((num_rows, num_columns), dtype=np.float32)
y = np.empty(num_rows, dtype=int)

for row_index, (context, target) in enumerate(pairs):
    context_to_sparse_row(row_index, context, X, word_to_id, context_size)
    y[row_index] = target_to_id[target]

X = X.tocsr()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Nonzero values in X:", X.nnz)
print("Density:", round(X.nnz / (X.shape[0] * X.shape[1]), 6))

## 7. Review the Prepared Dataset

Each row means:

```text
X[i] -> y[i]
```

The model sees the context features in `X`.

The correct next-word class is in `y`.

In [ ]:
for index in range(10):
    context, target = pairs[index]
    target_id = y[index]
    print("Example", index)
    print("Context:", context)
    print("Target word:", target)
    print("Target class ID:", target_id)
    print("---")

## 8. Train/Test Split

We split the data into:

- training set: used to train the model,
- test set: used to evaluate the model.

We use `stratify=y` so the target-word distribution is similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 9. Introduce `MLPClassifier`

`MLPClassifier` is a simple Feedforward Neural Network from scikit-learn.

Parameters used here:

- `hidden_layer_sizes=(96,)`: one hidden layer with 96 neurons
- `activation="relu"`: hidden-layer activation function
- `solver="adam"`: optimizer for larger datasets
- `batch_size=256`: trains on mini-batches
- `early_stopping=True`: stops if validation performance stops improving
- `random_state=42`: reproducible results

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(96,),
    activation="relu",
    solver="adam",
    batch_size=256,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42,
    max_iter=40,
)

print(model)

## 10. Train the FNN Model

Training means learning a mapping:

```text
context features -> next-word class
```

This cell may take a little longer than the toy examples because the dataset is larger.

In [ ]:
model.fit(X_train, y_train)

print("Training complete.")
print("Training iterations:", model.n_iter_)

## 11. Evaluate the Model

We evaluate with:

### Exact accuracy

The prediction is correct only if the top predicted word exactly matches the true next word.

### Top-5 accuracy

The prediction is counted as correct if the true word appears in the model's top 5 predicted words.

Top-5 accuracy is useful for language modeling because several next words may be reasonable.

In [ ]:
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

test_probabilities = model.predict_proba(X_test)
test_top5_accuracy = top_k_accuracy_score(
    y_test,
    test_probabilities,
    k=5,
    labels=model.classes_,
)

print("Training exact accuracy:", round(train_accuracy, 4))
print("Test exact accuracy:", round(test_accuracy, 4))
print("Test top-5 accuracy:", round(test_top5_accuracy, 4))

## 12. Compare with a Simple Baseline

A simple baseline always predicts the most common target word in the training set.

If our FNN is better than this baseline, it means the model learned something from the context.

In [ ]:
training_target_counts = Counter(y_train)
majority_class_id, majority_count = training_target_counts.most_common(1)[0]
majority_word = id_to_target[majority_class_id]
baseline_accuracy = majority_count / len(y_train)

print("Most common target word:", majority_word)
print("Baseline accuracy:", round(baseline_accuracy, 4))
print("FNN test exact accuracy:", round(test_accuracy, 4))
print("FNN test top-5 accuracy:", round(test_top5_accuracy, 4))

## 13. Inspect Test Predictions

Now compare true next words with predicted next words.

In [ ]:
test_predictions = model.predict(X_test)

for index in range(15):
    true_word = id_to_target[y_test[index]]
    predicted_word = id_to_target[test_predictions[index]]
    print("True:", true_word, "| Predicted:", predicted_word)

## 14. Predict One Next Word

To predict one next word:

1. Choose a context.
2. Convert it to a sparse feature row.
3. Use `model.predict()`.
4. Convert the predicted class ID back to a word.

In [ ]:
def context_to_vector(context, word_to_id, context_size):
    vector = lil_matrix((1, context_size * len(word_to_id)), dtype=np.float32)
    context_to_sparse_row(0, context, vector, word_to_id, context_size)
    return vector.tocsr()


context = ["the", "space"]

x_input = context_to_vector(context, word_to_id, context_size)
predicted_class_id = model.predict(x_input)[0]
predicted_word = id_to_target[predicted_class_id]

print("Context:", context)
print("Predicted next word:", predicted_word)

## 15. Inspect Prediction Probabilities

`predict_proba()` shows probability scores for each target word class.

We display the top 5 predictions.

In [ ]:
def show_top_predictions(context, model, top_k=5):
    x_input = context_to_vector(context, word_to_id, context_size)
    probabilities = model.predict_proba(x_input)[0]
    top_indexes = np.argsort(probabilities)[::-1][:top_k]

    print("Context:", context)
    for probability_index in top_indexes:
        class_id = model.classes_[probability_index]
        word = id_to_target[class_id]
        probability = probabilities[probability_index]
        print(word, "->", round(float(probability), 4))


show_top_predictions(["the", "space"], model, top_k=5)

## 16. Create a Helper Function

The helper function checks:

- context size,
- unknown context words.

In [ ]:
def predict_next_word(context, model):
    context = [word.lower() for word in context]

    if len(context) != context_size:
        raise ValueError(f"Expected {context_size} context words, got {len(context)}.")

    unknown_words = [word for word in context if word not in word_to_id]
    if unknown_words:
        raise ValueError(f"Unknown context words: {unknown_words}")

    x_input = context_to_vector(context, word_to_id, context_size)
    predicted_class_id = model.predict(x_input)[0]

    return id_to_target[predicted_class_id]


print(["the", "space"], "->", predict_next_word(["the", "space"], model))
print(["the", "car"], "->", predict_next_word(["the", "car"], model))

## 17. Test Different Contexts

These contexts come from common words in the real dataset.

In [ ]:
test_contexts = [
    ["the", "space"],
    ["the", "car"],
    ["the", "team"],
    ["the", "game"],
    ["a", "new"],
    ["i", "think"],
    ["it", "is"],
    ["this", "is"],
]

for context in test_contexts:
    print(context, "->", predict_next_word(context, model))

## 18. Text Generation

To generate text:

1. Start with a seed context.
2. Predict the next word.
3. Add the predicted word.
4. Move the context window forward.
5. Repeat.

This uses greedy generation: always choose the most likely next word.

In [ ]:
def generate_text_greedy(seed, n_words, model):
    generated = seed.lower().split()

    if len(generated) < context_size:
        raise ValueError(f"Seed must contain at least {context_size} words.")

    for word in generated:
        if word not in word_to_id:
            raise ValueError(f"Unknown word in seed: {word}")

    for _ in range(n_words):
        context = generated[-context_size:]
        next_word = predict_next_word(context, model)
        generated.append(next_word)

    return " ".join(generated)


print(generate_text_greedy("the space", 15, model))
print(generate_text_greedy("the car", 15, model))
print(generate_text_greedy("i think", 15, model))

## 19. Random Sampling Generation

Greedy generation can become repetitive.

Random sampling chooses from the probability distribution, which can create more variety.

We use a fixed random seed so the result is reproducible.

In [ ]:
def sample_next_word(context, model, rng):
    x_input = context_to_vector(context, word_to_id, context_size)
    probabilities = model.predict_proba(x_input)[0]
    sampled_class_id = rng.choice(model.classes_, p=probabilities)
    return id_to_target[int(sampled_class_id)]


def generate_text_sampling(seed, n_words, model, random_seed=7):
    rng = np.random.default_rng(random_seed)
    generated = seed.lower().split()

    if len(generated) < context_size:
        raise ValueError(f"Seed must contain at least {context_size} words.")

    for word in generated:
        if word not in word_to_id:
            raise ValueError(f"Unknown word in seed: {word}")

    for _ in range(n_words):
        context = generated[-context_size:]
        next_word = sample_next_word(context, model, rng)
        generated.append(next_word)

    return " ".join(generated)


print("Greedy:")
print(generate_text_greedy("the space", 15, model))

print("\nSampling:")
print(generate_text_sampling("the space", 15, model, random_seed=7))

## 20. Generated Text Quality

Reflection questions:

1. Does the generated text sound like real discussion text?
2. Does it become repetitive?
3. Does it stay on topic?
4. Which is more interesting: greedy generation or sampling?
5. Why is exact accuracy still difficult in real-world language modeling?

Real-world text is harder than toy text because many next words can be reasonable.

## 21. Limitations of FNN Language Models

Even with more data, an FNN language model still has limitations.

### Fixed context size

The model only sees two previous words.

### Weak long-term memory

It does not naturally remember earlier sentences or paragraphs.

### Large vocabulary problem

Real-world text has many words, so we had to limit the vocabulary and target classes.

### Repetition

Greedy generation can repeat common patterns.

### Limited meaning

The model learns statistical patterns, not deep understanding.

## 22. Bridge to RNNs and Transformers

The FNN is useful because it teaches the full workflow:

```text
text -> tokens -> vocabulary -> features -> model -> prediction
```

But more advanced language models handle sequence better.

### RNNs

RNNs process sequences step by step and can carry information forward.

### Transformers

Transformers use attention to look at context more powerfully.

Modern large language models are based on Transformer architectures.

## Session 3 Summary

In this session, we:

- used a larger real-world text dataset,
- built context-target pairs,
- used sparse one-hot context features,
- trained `MLPClassifier`,
- measured exact accuracy and top-5 accuracy,
- compared against a majority baseline,
- predicted next words,
- generated text,
- discussed why real-world language modeling is harder than toy examples.